<a href="https://colab.research.google.com/github/BassemRamdan/AI-Resume-Intelligence/blob/main/notebooks/05_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 5: Classical Machine Learning Baseline

**Goal:**
- Establish a strong baseline for the Resume Classification Task using classical ML (TF-IDF + Logistic Regression/SVM).
- Use the Stratified Splits from Phase 2 to ensure no Data Leakage and fair evaluation of imbalanced classes.
- Report Accuracy, Precision, Recall, Macro F1, and plot the Confusion Matrix.

In [ ]:
!pip install pandas scikit-learn matplotlib seaborn huggingface_hub

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# 1. Load Data
# Load the splits from Phase 2
splits_df = pd.read_csv("resume_metadata_split.csv")
# Load the NLP processed text from Phase 4
nlp_df = pd.read_csv("nlp_processed_resumes.csv")

# Merge them based on filename to get the 'split' column alongside the 'lemmatized_text'
df = pd.merge(nlp_df, splits_df[['filename', 'split']], on='filename', how='inner')

train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'val']
test_df = df[df['split'] == 'test']

print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")

In [ ]:
# 2. TF-IDF Feature Extraction
print("Extracting TF-IDF Features...")
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

# Fit on train, transform train/val/test
X_train = tfidf.fit_transform(train_df['lemmatized_text'].fillna(""))
X_val = tfidf.transform(val_df['lemmatized_text'].fillna(""))
X_test = tfidf.transform(test_df['lemmatized_text'].fillna(""))

y_train = train_df['category']
y_val = val_df['category']
y_test = test_df['category']

print(f"Vocabulary size: {len(tfidf.vocabulary_)}")

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    print(f"\n--- Evaluating {model_name} ---")
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    # We focus on MACRO avg due to heavy class imbalance
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
    
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision (Macro): {prec:.4f}")
    print(f"Recall (Macro):    {rec:.4f}")
    print(f"F1 Score (Macro):  {f1:.4f}")
    
    return y_pred

def plot_confusion_matrix(y_true, y_pred, classes, title):
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(title, fontsize=16)
    plt.ylabel('Actual Category')
    plt.xlabel('Predicted Category')
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

In [ ]:
# 3. Logistic Regression Baseline
print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train, y_train)

lr_preds = evaluate_model(lr_model, X_test, y_test, "Logistic Regression")

In [ ]:
# 4. Linear SVM Baseline
print("Training Linear SVM...")
svm_model = LinearSVC(class_weight='balanced', random_state=42, max_iter=2000)
svm_model.fit(X_train, y_train)

svm_preds = evaluate_model(svm_model, X_test, y_test, "Linear SVM")

In [ ]:
# 5. Confusion Matrix for the best model (SVM usually performs slightly better on TF-IDF)
classes = sorted(y_test.unique())
plot_confusion_matrix(y_test, svm_preds, classes, "Confusion Matrix: Linear SVM (TF-IDF)")

print("\nDetailed Classification Report (SVM):")
print(classification_report(y_test, svm_preds))